# 11 — Design the next dataset project safely

**Estimated time:** 35 minutes<br>
**Prerequisites:** 10 — Capstone model versus hybrid<br>
**Learner-produced evidence:** a dataset-review checklist and task-specific evaluation plan

## Learning objectives

- Treat optional datasets as a review queue, not pre-approved inputs.
- Narrow the task before adding multilingual, extraction, transformation, or code-generation complexity.
- Match evaluation to the behavior instead of reusing intent metrics blindly.

This notebook is a teaching interface over the reusable code in `src/`.
It uses only prepared local files. Run `make prepare-flight` before the trip;
no cell installs packages or downloads data.


In [ ]:
import sys
from importlib import import_module
from pathlib import Path

current = Path.cwd().resolve()
project_root = None
for candidate in (current, *current.parents):
    direct = candidate
    nested = candidate / "examples" / "local-finetuning"
    if (direct / "src" / "aai_local_finetuning").is_dir():
        project_root = direct
        break
    if (nested / "src" / "aai_local_finetuning").is_dir():
        project_root = nested
        break
if project_root is None:
    raise RuntimeError(
        "Cannot locate examples/local-finetuning. Open this notebook from the "
        "repository, or run `make notebook` from the repository root."
    )

expected_python = (project_root / ".venv" / "bin" / "python").resolve()
active_python = Path(sys.executable).resolve()
if not expected_python.is_file() or active_python != expected_python:
    raise RuntimeError(
        "Wrong notebook kernel. Run `make notebook` from the repository root, "
        "then select 'AAI Local Fine-Tuning (offline)'. "
        f"Active Python: {active_python}; expected: {expected_python}"
    )

source_root = str(project_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

enable_offline_environment = import_module(
    "aai_local_finetuning.offline"
).enable_offline_environment
enable_offline_environment()

## Why these are plans, not executable labs yet

The optional Kaggle datasets are not cached or verified in this project.
Their current schema, license, access, source composition, sensitive-data
risk, and redistribution terms must be reviewed before code or claims are
added. Offline study should never fill those gaps with assumptions.


In [ ]:
candidates = [
    {
        "project": "multilingual support tickets",
        "first_task": "English-only queue/priority/type/review routing",
        "evaluation": "per-field metrics, imbalance and queue/priority slices",
        "status": "unsuitable until current source review is complete",
    },
    {
        "project": "invoice extraction",
        "first_task": "text-only fields only if reliable OCR text exists",
        "evaluation": "field exactness/F1, normalization, hallucinated fields",
        "status": "unsuitable until modality, schema, and rights are verified",
    },
    {
        "project": "prompt transformation",
        "first_task": "required prompt components and structured output",
        "evaluation": "rubric components plus calibrated human review",
        "status": "optional; open-ended scoring is less deterministic",
    },
    {
        "project": "MiniZinc generation",
        "first_task": "small natural-language-to-program problems",
        "evaluation": "parse, compile, execute, constraints, objective, runtime",
        "status": "advanced; may exceed tiny-model capability",
    },
]
candidates

## Required source review

A current review must record title, owner, URL, license, permitted use,
redistribution, size, formats, confirmed columns, languages, labels,
missingness, duplicates, sensitive information, human/synthetic origin,
accessibility, and access date. An unclear license means unsuitable until
the learner verifies it directly.


In [ ]:
required_review_fields = (
    "title",
    "owner",
    "current_url",
    "license",
    "permitted_use",
    "redistribution",
    "size",
    "formats",
    "columns",
    "languages",
    "label_quality",
    "missing_values",
    "duplicate_rate",
    "sensitive_information",
    "record_origin",
    "accessible",
    "accessed_on",
)
required_review_fields

## Exercise — draft, but do not invent, a review

Choose one project. Fill only facts you have verified later while online;
leave unknowns as `None`. The suitability gate must remain false whenever
the license, schema, access, or modality is unknown.


In [ ]:
dataset_review = {field: None for field in required_review_fields}
dataset_review["title"] = "<verify current title online>"
dataset_review["accessible"] = None
blocking_fields = ("license", "columns", "accessible")
suitable_for_lab = all(dataset_review[field] for field in blocking_fields)
{
    "review": dataset_review,
    "suitable_for_lab": suitable_for_lab,
    "decision": (
        "continue to adapter design"
        if suitable_for_lab
        else "unsuitable until direct verification"
    ),
}

## Exercise — choose task-shaped metrics

Select one task and add a metric that catches a failure ordinary exact
match would miss. For images, remember that a text-only model cannot see
pixels without a separate OCR or multimodal stage.


In [ ]:
evaluation_plan = {
    "task": "invoice text extraction",
    "principal_metrics": [
        "field-level precision/recall/F1",
        "normalized date and currency exactness",
        "hallucinated-field rate",
        "schema validity",
    ],
    "human_review_rule": (
        "route missing, conflicting, or low-confidence critical fields"
    ),
    "modality_boundary": (
        "requires reliable OCR text; a tiny text model cannot read images"
    ),
}
evaluation_plan

**Hint:** the output contract determines the evaluator. Classification,
extraction, open-ended transformation, and executable code require
different evidence.


## Final checkpoint

You have completed a full local lifecycle and can now design the next
project without assuming that a public dataset is licensed, suitable,
text-only, balanced, clean, or evaluable in the same way.

Revisit `00_start_here.ipynb` with a new evidence question, version the
source contract, and create a new untouched evaluation boundary.
